<a href="https://colab.research.google.com/github/AyaAbdElNaem/AI_Tools/blob/main/Lab7_Embeddings_and_Semantic_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 7: Embeddings and Semantic Retrieval
## From Keyword Matching to Meaning-Based Search

In the previous lab, we built lexical retrievers using Bag-of-Words, TF-IDF, cosine similarity, and BM25.

Those methods are powerful, but they mainly depend on **word overlap**.

In this lab, we move from **keyword search** to **semantic search**.

The main question is:

> How can a retrieval system understand that two texts are related even when they do not use the same words?

Example:

```text
Query:    "How do I get my money back?"
Document: "Students can request a refund within 14 days."
```

The query says **money back**.
The document says **refund**.

A lexical retriever may miss this match because the exact words are different.

An embedding-based retriever should rank this document highly because the **meaning** is similar.


## Learning Outcomes

By the end of this lab, you should be able to explain and implement:

| Topic | What you must understand |
|---|---|
| Lexical retrieval failure | Why TF-IDF and BM25 can fail when words differ |
| Embedding | A dense numerical vector representing text meaning |
| Sparse vs dense vectors | Why TF-IDF and embeddings are structurally different |
| Sentence embedding model | How text is converted into a fixed-length vector |
| Query embedding | Why query and documents must use the same embedding model |
| Cosine similarity | How embedding vectors are compared |
| Semantic retrieval | How to retrieve top-k documents by meaning |
| Evaluation | How to compare TF-IDF, BM25, and embeddings fairly |
| Hybrid retrieval | Why combining lexical and semantic search is often stronger |
| Failure analysis | Why embeddings are useful but not always safe |
| FAISS indexing | How vector indexes accelerate search for larger corpora |


## The Big Picture

A semantic retrieval system follows this pipeline:

```text
Documents
→ embedding model
→ document embeddings
→ user query
→ query embedding
→ similarity scores
→ top-k retrieved documents
→ evaluation
```

A professional retrieval engineer does not simply ask:

> Did the model return something?

A professional asks:

> Did the retriever return the right documents, in the right order, for the right reason?


## Important Design Rule for This Lab

We are not asking a language model to generate answers yet.

Why?

Because if retrieval is weak, the final generated answer will also be weak.

Before building a RAG pipeline, you must understand:

```text
retrieval quality → context quality → answer quality
```


# Section 1 — Install Required Packages

This notebook uses:

- `scikit-learn` for TF-IDF, cosine similarity, and evaluation helpers.
- `rank-bm25` for BM25 lexical retrieval.
- `sentence-transformers` for dense text embeddings.
- `faiss-cpu` in the optional vector indexing section.

Run the installation cell once if these packages are not already installed.

If you are using Google Colab, run it normally.
If you are using a local environment, install the packages in your selected Python environment.


In [ ]:
# Run this cell if the packages are not installed in your environment.
# In Google Colab, uncomment and run the following line:

!pip install -q sentence-transformers rank-bm25 faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 56.6 MB/s eta 0:00:00


# Section 2 — Import Libraries

We import the tools needed for the complete retrieval experiment.


In [ ]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 180)


## What Each Library Does

### `re`
Used for simple tokenization with regular expressions.

### `numpy`
Used for numerical arrays, score normalization, ranking, and vector operations.

### `pandas`
Used to display retrieval results and evaluation metrics in readable tables.

### `TfidfVectorizer`
Converts documents into sparse TF-IDF vectors.
This gives us a lexical baseline.

### `cosine_similarity`
Measures similarity between vectors using the cosine of the angle between them.

### `BM25Okapi`
Implements BM25 ranking using the `rank-bm25` library.
We use it directly instead of implementing BM25 manually.

### `SentenceTransformer`
Loads a pretrained embedding model that converts text into dense vectors.


# Section 3 — Create a Neutral Knowledge Base

We will use a small neutral knowledge base about university services.

Each row is treated as one document.

In real systems, documents may come from:

- help-center pages,
- policies,
- FAQs,
- PDFs,
- manuals,
- product documentation,
- customer-support records.


In [ ]:
documents = [
    "Students can request a refund within 14 days of payment by submitting the online finance form.",
    "The library is open from 8 AM to 8 PM on weekdays and from 10 AM to 4 PM on Saturdays.",
    "The registration office handles enrollment, course changes, section transfers, and class withdrawal requests.",
    "Technical support can help students reset account passwords and recover access to the student portal.",
    "Counseling services provide confidential academic, emotional, and personal support for students.",
    "The exam timetable is published on the student portal two weeks before the assessment period begins.",
    "Student housing applications must be submitted before July 15 through the residence services system.",
    "The cafeteria serves breakfast, lunch, snacks, and drinks during normal campus operating hours.",
    "Campus shuttle buses transport students between the main building, labs, library, and residence halls.",
    "The scholarship office reviews financial aid applications and merit-based funding requests.",
    "Laboratory safety orientation is required before students may enter chemistry, biology, or engineering labs.",
    "The medical clinic treats minor injuries, provides first aid, and refers urgent cases to nearby hospitals.",
    "Thesis drafts must be uploaded to the research portal before the department submission deadline.",
    "Printing services are available in the library, computer lab, and student services center.",
    "Lost student ID cards can be replaced at the student services desk after identity verification.",
    "Projector and meeting-room bookings are managed by the facilities office through an online reservation system.",
    "The career center helps students prepare resumes, practice interviews, and find internship opportunities.",
    "Lost and found items are stored at campus security for thirty days before disposal.",
    "Parking permits are issued by campus security after vehicle registration and payment confirmation.",
    "The international office supports visa letters, travel documentation, and immigration-related student questions."
]

corpus_df = pd.DataFrame({
    "document_id": range(len(documents)),
    "document": documents
})

corpus_df


,document_id,document
0,0,Students can request a refund within 14 days of payment by submitting the online finance form.
1,1,The library is open from 8 AM to 8 PM on weekdays and from 10 AM to 4 PM on Saturdays.
2,2,"The registration office handles enrollment, course changes, section transfers, and class withdrawal requests."
3,3,Technical support can help students reset account passwords and recover access to the student portal.
4,4,"Counseling services provide confidential academic, emotional, and personal support for students."
5,5,The exam timetable is published on the student portal two weeks before the assessment period begins.
6,6,Student housing applications must be submitted before July 15 through the residence services system.
7,7,"The cafeteria serves breakfast, lunch, snacks, and drinks during normal campus operating hours."
8,8,"Campus shuttle buses transport students between the main building, labs, library, and residence halls."
9,9,The scholarship office reviews financial aid applications and merit-based funding requests.


## Why This Dataset Is Designed This Way

The documents include different retrieval patterns:

| Pattern | Example |
|---|---|
| Synonym/paraphrase | "money back" should match "refund" |
| Operational intent | "change my classes" should match registration/course changes |
| Support language | "forgot login" should match password/account recovery |
| Time-sensitive facts | "exam schedule" should match timetable publication |
| Exact details | numbers like 14 days and July 15 matter |

This allows us to test where lexical search works and where semantic search is stronger.


# Section 4 — Create Queries and Ground Truth

A query is the user's search input.

Ground truth is the manually defined list of correct document IDs for each query.

Without ground truth, you cannot evaluate retrieval quality objectively.


In [ ]:
ground_truth = {
    "How do I get my money back?": [0],
    "Where can I change my classes?": [2],
    "I forgot my login details": [3],
    "When are tests announced?": [5],
    "Where can I get emotional support?": [4],
    "Can I live on campus?": [6],
    "Where do I report a small injury?": [11],
    "How can I replace my badge?": [14],
    "Where can I find my missing backpack?": [17],
    "How do I travel between campus buildings?": [8],
    "Where can I get help finding an internship?": [16],
    "Who helps with visa documents?": [19]
}

queries_df = pd.DataFrame({
    "query": list(ground_truth.keys()),
    "relevant_document_ids": list(ground_truth.values())
})

queries_df


,query,relevant_document_ids
0,How do I get my money back?,[0]
1,Where can I change my classes?,[2]
2,I forgot my login details,[3]
3,When are tests announced?,[5]
4,Where can I get emotional support?,[4]
5,Can I live on campus?,[6]
6,Where do I report a small injury?,[11]
7,How can I replace my badge?,[14]
8,Where can I find my missing backpack?,[17]
9,How do I travel between campus buildings?,[8]


## Important Ground Truth Rule

Ground truth should be created by a human who understands the task.

For example:

```text
Query: "How do I get my money back?"
Relevant document: 0
```

Because document 0 says:

```text
"Students can request a refund within 14 days..."
```

The words are different, but the intent is the same.


# Section 5 — Retrieval Metrics

Before building semantic retrieval, we need the same metrics for all retrieval systems.

We will compare:

1. TF-IDF retrieval
2. BM25 retrieval
3. Embedding retrieval
4. Hybrid retrieval

Using the same metrics makes the comparison fair.


## Precision@K


## Recall@K



## Hit Rate@K



## Reciprocal Rank and MRR



In [ ]:
def precision_at_k(retrieved_ids, relevant_ids, k):

    retrieved_at_k = retrieved_ids[:k]
    relevant_retrieved = set(retrieved_at_k).intersection(set(relevant_ids))
    return len(relevant_retrieved) / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    """
    Calculate Recall@K.
    """
    retrieved_at_k = retrieved_ids[:k]
    relevant_retrieved = set(retrieved_at_k).intersection(set(relevant_ids))
    return len(relevant_retrieved) / len(relevant_ids)


def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    """
    Calculate Hit Rate@K.
    """
    retrieved_at_k = retrieved_ids[:k]
    relevant_retrieved = set(retrieved_at_k).intersection(set(relevant_ids))
    return 1 if len(relevant_retrieved) > 0 else 0


def reciprocal_rank(retrieved_ids, relevant_ids):
    """
    Calculate reciprocal rank for one query.
    """
    for rank, document_id in enumerate(retrieved_ids, start=1):
        if document_id in relevant_ids:
            return 1 / rank
    return 0


# Section 6 — Lexical Baseline 1: TF-IDF Retriever

Before we use embeddings, we need a baseline.

A baseline is a simple system used for comparison.

If embeddings cannot beat a simple baseline, then the embedding system may not be useful for that task.

TF-IDF is a lexical method because it depends on exact words.


In [ ]:
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(documents)

print("TF-IDF matrix shape:", tfidf_matrix.shape)


TF-IDF matrix shape: (20, 183)


## What `fit_transform()` Means Here

`fit_transform(documents)` does two things:

1. `fit`
   - learns the vocabulary,
   - calculates document frequency,
   - calculates IDF values.

2. `transform`
   - converts every document into a TF-IDF vector.

After this, every document is represented as a sparse numerical vector.


In [ ]:
def retrieve_top_k_tfidf(query, vectorizer, document_matrix, documents, k=3):
    """
    Retrieve top-k documents using TF-IDF vectors and cosine similarity.
    """
    query_vector = vectorizer.transform([query])

    scores = cosine_similarity(query_vector, document_matrix).flatten()

    results = pd.DataFrame({
        "document_id": range(len(documents)),
        "document": documents,
        "score": scores
    })

    results = results.sort_values(by="score", ascending=False)

    return results.head(k).reset_index(drop=True)


In [ ]:
retrieve_top_k_tfidf(
    query="How do I get my money back?",
    vectorizer=tfidf_vectorizer,
    document_matrix=tfidf_matrix,
    documents=documents,
    k=5
)


,document_id,document,score
0,0,Students can request a refund within 14 days of payment by submitting the online finance form.,0.0
1,1,The library is open from 8 AM to 8 PM on weekdays and from 10 AM to 4 PM on Saturdays.,0.0
2,18,Parking permits are issued by campus security after vehicle registration and payment confirmation.,0.0
3,17,Lost and found items are stored at campus security for thirty days before disposal.,0.0
4,16,"The career center helps students prepare resumes, practice interviews, and find internship opportunities.",0.0


## Ruthless Interpretation

If TF-IDF ranks the refund document low, the reason is usually vocabulary mismatch.

The query uses:

```text
money back
```

The document uses:

```text
refund
```

TF-IDF does not know that these phrases are semantically related unless the words overlap.


# Section 7 — Lexical Baseline 2: BM25 Using a Library

BM25 is also a lexical retrieval method.

It is usually stronger than raw TF-IDF for keyword search because it considers:

1. term frequency,
2. inverse document frequency,
3. document length normalization,
4. term-frequency saturation.

In this lab, we use `rank-bm25` directly.



In [ ]:
def simple_tokenize(text):
    """
    Convert text into lowercase alphanumeric tokens.

    Example:
    "The library opens at 8 AM."
    becomes:
    ["the", "library", "opens", "at", "8", "am"]
    """
    return re.findall(r"[a-z0-9]+", text.lower())


def build_bm25_index(documents):
    """
    Build a BM25 index using the rank-bm25 library.

    Returns:
    bm25:
        A BM25Okapi object.

    tokenized_documents:
        The tokenized document collection.
    """
    tokenized_documents = [
        simple_tokenize(document)
        for document in documents
    ]

    bm25 = BM25Okapi(tokenized_documents)

    return bm25, tokenized_documents


def retrieve_top_k_bm25(query, documents, bm25, k=3):
    """
    Retrieve top-k documents using BM25.
    """
    tokenized_query = simple_tokenize(query)

    scores = bm25.get_scores(tokenized_query)

    results = pd.DataFrame({
        "document_id": range(len(documents)),
        "document": documents,
        "score": scores
    })

    results = results.sort_values(by="score", ascending=False)

    return results.head(k).reset_index(drop=True)


In [ ]:
bm25, tokenized_documents = build_bm25_index(documents)

retrieve_top_k_bm25(
    query="How do I get my money back?",
    documents=documents,
    bm25=bm25,
    k=5
)


,document_id,document,score
0,0,Students can request a refund within 14 days of payment by submitting the online finance form.,0.0
1,1,The library is open from 8 AM to 8 PM on weekdays and from 10 AM to 4 PM on Saturdays.,0.0
2,18,Parking permits are issued by campus security after vehicle registration and payment confirmation.,0.0
3,17,Lost and found items are stored at campus security for thirty days before disposal.,0.0
4,16,"The career center helps students prepare resumes, practice interviews, and find internship opportunities.",0.0


## What `BM25Okapi` Does

`BM25Okapi` builds a BM25 retriever from tokenized documents.

The input must be a list of token lists:

```python
[
    ["students", "can", "request", "a", "refund"],
    ["the", "library", "is", "open"]
]
```

The method:

```python
bm25.get_scores(tokenized_query)
```

returns one BM25 score per document.

Higher score means the document is more relevant according to BM25.


# Section 8 — Evaluate Lexical Retrievers

Now we evaluate TF-IDF and BM25 on all queries.

This gives us a baseline before adding embeddings.


In [ ]:
def evaluate_retriever(retriever_name, retrieval_function, ground_truth, k=3, max_results=None):
    """
    Evaluate any retriever that returns a DataFrame with a document_id column.

    retriever_name:
        Name of the retrieval method.

    retrieval_function:
        A function that accepts query and k, then returns ranked results.

    ground_truth:
        Dictionary mapping query to relevant document IDs.
    """
    if max_results is None:
        max_results = len(documents)

    rows = []

    for query, relevant_ids in ground_truth.items():
        results = retrieval_function(query, max_results)
        retrieved_ids = results["document_id"].tolist()

        rows.append({
            "retriever": retriever_name,
            "query": query,
            "retrieved_ids": retrieved_ids[:k],
            "relevant_ids": relevant_ids,
            f"precision@{k}": precision_at_k(retrieved_ids, relevant_ids, k),
            f"recall@{k}": recall_at_k(retrieved_ids, relevant_ids, k),
            f"hit_rate@{k}": hit_rate_at_k(retrieved_ids, relevant_ids, k),
            "reciprocal_rank": reciprocal_rank(retrieved_ids, relevant_ids)
        })

    return pd.DataFrame(rows)


In [ ]:
tfidf_eval = evaluate_retriever(
    retriever_name="TF-IDF",
    retrieval_function=lambda query, k: retrieve_top_k_tfidf(
        query=query,
        vectorizer=tfidf_vectorizer,
        document_matrix=tfidf_matrix,
        documents=documents,
        k=k
    ),
    ground_truth=ground_truth,
    k=3,
    max_results=len(documents)
)

bm25_eval = evaluate_retriever(
    retriever_name="BM25",
    retrieval_function=lambda query, k: retrieve_top_k_bm25(
        query=query,
        documents=documents,
        bm25=bm25,
        k=k
    ),
    ground_truth=ground_truth,
    k=3,
    max_results=len(documents)
)

pd.concat([tfidf_eval, bm25_eval], ignore_index=True)


,retriever,query,retrieved_ids,relevant_ids,precision@3,recall@3,hit_rate@3,reciprocal_rank
0,TF-IDF,How do I get my money back?,"[0, 1, 18]",[0],0.333333,1.0,1,1.000000
1,TF-IDF,Where can I change my classes?,"[3, 14, 0]",[2],0.000000,0.0,0,0.052632
2,TF-IDF,I forgot my login details,"[0, 1, 18]",[3],0.000000,0.0,0,0.055556
3,TF-IDF,When are tests announced?,"[13, 18, 17]",[5],0.000000,0.0,0,0.062500
4,TF-IDF,Where can I get emotional support?,"[4, 3, 14]",[4],0.333333,1.0,1,1.000000
5,TF-IDF,Can I live on campus?,"[1, 5, 3]",[6],0.000000,0.0,0,0.058824
6,TF-IDF,Where do I report a small injury?,"[0, 1, 18]",[11],0.000000,0.0,0,0.100000
7,TF-IDF,How can I replace my badge?,"[3, 14, 0]",[14],0.333333,1.0,1,0.500000
8,TF-IDF,Where can I find my missing backpack?,"[16, 3, 14]",[17],0.000000,0.0,0,0.142857
9,TF-IDF,How do I travel between campus buildings?,"[8, 19, 18]",[8],0.333333,1.0,1,1.000000


In [ ]:
lexical_summary = pd.concat([tfidf_eval, bm25_eval], ignore_index=True).groupby("retriever")[[
    "precision@3",
    "recall@3",
    "hit_rate@3",
    "reciprocal_rank"
]].mean()

lexical_summary


,precision@3,recall@3,hit_rate@3,reciprocal_rank
retriever,,,,
BM25,0.166667,0.5,0.5,0.456031
TF-IDF,0.166667,0.5,0.5,0.456031


## What to Look For

Check which queries failed.

A lexical retriever usually fails when:

- the query uses synonyms,
- the document uses different wording,
- the user expresses intent indirectly,
- abbreviations differ from full terms,
- the query is conversational but the document is formal.

That is exactly why semantic retrieval exists.


# Section 9 — What Is an Embedding?

An embedding is a dense numerical vector that represents text meaning.

A sentence embedding model converts text into numbers.

Example:

```text
"refund policy" → [0.12, -0.04, 0.31, ..., 0.08]
```

The individual numbers are usually not directly interpretable.

What matters is the position of the vector compared with other vectors.

Texts with similar meanings should produce vectors that are close to each other.


## Sparse Vectors vs Dense Vectors

| Property | TF-IDF Vector | Embedding Vector |
|---|---|---|
| Type | Sparse | Dense |
| Meaning of dimensions | Usually vocabulary words | Learned semantic features |
| Most values | Zero | Non-zero |
| Handles synonyms well? | Usually no | Often yes |
| Interpretable? | More interpretable | Less interpretable |
| Good for exact words/codes? | Strong | Can be weak |
| Good for paraphrases? | Weak | Stronger |

Sparse TF-IDF vector:

```text
[0, 0, 0.43, 0, 0, 0, 0.21, 0, ...]
```

Dense embedding vector:

```text
[0.12, -0.07, 0.44, 0.03, -0.15, ...]
```


# Section 10 — Load a Sentence Embedding Model

We will use a small pretrained sentence transformer model.

The model converts sentences and paragraphs into fixed-length vectors.

The first time this cell runs, the model may be downloaded from the internet.


In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")


## What Happens Inside a Sentence Embedding Model?

A simplified view:

```text
text
→ tokenizer
→ transformer encoder
→ pooling
→ final embedding vector
```

### Tokenizer
Splits text into model-readable pieces.

### Transformer encoder
Processes the text and creates contextual token representations.

### Pooling
Combines token-level representations into one vector for the whole sentence or paragraph.

### Embedding vector
The final dense vector used for similarity search.


# Section 11 — Encode Documents

Now we convert every document into an embedding vector.


In [ ]:
document_embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Document embeddings shape:", document_embeddings.shape)


Document embeddings shape: (20, 384)


## Understanding the Shape

If the output is:

```text
(20, 384)
```

That means:

```text
20  = number of documents
384 = embedding dimensions
```

Each document is now represented by a vector of length 384.

### Why `normalize_embeddings=True`?

Normalization scales each vector to unit length.

For normalized vectors:

```text
cosine similarity = dot product
```

This is useful because it makes similarity search simpler and faster.


In [ ]:
first_embedding = document_embeddings[0]

print("First embedding length:", len(first_embedding))
print("First 10 values:", first_embedding[:10])
print("Vector norm:", np.linalg.norm(first_embedding))


First embedding length: 384
First 10 values: [-0.03394912  0.04964762  0.04764685  0.01189213  0.0118311  -0.05036814
 -0.05492017 -0.01455906  0.01258931  0.06476928]
Vector norm: 1.0


## Can We Interpret Each Embedding Number?

Usually, no.

In TF-IDF, each column corresponds to a known vocabulary word.

In embeddings, each dimension is a learned feature.

The dimensions are useful mathematically, but they are not usually meaningful as standalone human-readable concepts.

So we inspect embeddings through:

- similarity scores,
- retrieval results,
- evaluation metrics,
- failure analysis.


# Section 12 — Encode a Query

The query must be embedded using the **same model** used for documents.

If documents and queries are embedded by different models, their vectors may not be comparable.


In [ ]:
query = "How do I get my money back?"

query_embedding = model.encode(
    [query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Query embedding shape:", query_embedding.shape)


Query embedding shape: (1, 384)


## Why Put the Query Inside a List?

We use:

```python
model.encode([query])
```

instead of:

```python
model.encode(query)
```

because we want a two-dimensional output:

```text
(number_of_queries, embedding_dimension)
```

This makes it easier to compare one query against many document vectors.


# Section 13 — Cosine Similarity for Embeddings

Cosine similarity measures how close two vectors are in direction.


For normalized embeddings, vector length is 1, so cosine similarity becomes equivalent to dot product.


In [ ]:
semantic_scores = cosine_similarity(
    query_embedding,
    document_embeddings
).flatten()

semantic_scores[:5]


array([ 0.4529754 , -0.04498605,  0.10176048,  0.17349079, -0.01428396],
      dtype=float32)

## How to Interpret Embedding Similarity Scores

Higher score means the query and document are semantically closer.

Typical interpretation:

| Score Range | Rough Meaning |
|---|---|
| close to 1 | very similar |
| around 0.4 to 0.7 | potentially related |
| around 0.0 to 0.3 | weakly related or unrelated |
| below 0 | usually dissimilar |

Do not use these thresholds blindly.

Similarity scores depend on:

- the embedding model,
- the dataset,
- the domain,
- the query style,
- document length.


# Section 14 — Build Semantic Search Results


In [ ]:
semantic_results = pd.DataFrame({
    "document_id": range(len(documents)),
    "document": documents,
    "semantic_score": semantic_scores
}).sort_values(by="semantic_score", ascending=False)

semantic_results.head(5)


,document_id,document,semantic_score
0,0,Students can request a refund within 14 days of payment by submitting the online finance form.,0.452975
3,3,Technical support can help students reset account passwords and recover access to the student portal.,0.173491
9,9,The scholarship office reviews financial aid applications and merit-based funding requests.,0.147708
14,14,Lost student ID cards can be replaced at the student services desk after identity verification.,0.116626
17,17,Lost and found items are stored at campus security for thirty days before disposal.,0.114521


## The Expected Aha Moment

The refund document should rank highly for:

```text
"How do I get my money back?"
```

Even though the document uses:

```text
"refund"
```

This is the practical difference between lexical retrieval and semantic retrieval.


# Section 15 — Build a Reusable Semantic Retriever


In [ ]:
def retrieve_top_k_semantic(query, model, documents, document_embeddings, k=3):
    """
    Retrieve top-k documents using dense embeddings and cosine similarity.
    """
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(query_embedding, document_embeddings).flatten()

    results = pd.DataFrame({
        "document_id": range(len(documents)),
        "document": documents,
        "score": scores
    })

    results = results.sort_values(by="score", ascending=False)

    return results.head(k).reset_index(drop=True)


In [ ]:
retrieve_top_k_semantic(
    query="Where can I change my classes?",
    model=model,
    documents=documents,
    document_embeddings=document_embeddings,
    k=5
)


,document_id,document,score
0,2,"The registration office handles enrollment, course changes, section transfers, and class withdrawal requests.",0.513728
1,13,"Printing services are available in the library, computer lab, and student services center.",0.283375
2,14,Lost student ID cards can be replaced at the student services desk after identity verification.,0.280015
3,16,"The career center helps students prepare resumes, practice interviews, and find internship opportunities.",0.279395
4,8,"Campus shuttle buses transport students between the main building, labs, library, and residence halls.",0.273851


## Semantic Retriever Pipeline

The function does this:

1. Receives a user query.
2. Converts the query into an embedding.
3. Compares the query embedding with all document embeddings.
4. Sorts documents by similarity score.
5. Returns the top-k documents.

This is the core retrieval step used before RAG generation.


# Section 16 — Evaluate Semantic Retriever


In [ ]:
semantic_eval = evaluate_retriever(
    retriever_name="Embeddings",
    retrieval_function=lambda query, k: retrieve_top_k_semantic(
        query=query,
        model=model,
        documents=documents,
        document_embeddings=document_embeddings,
        k=k
    ),
    ground_truth=ground_truth,
    k=3,
    max_results=len(documents)
)

semantic_eval


,retriever,query,retrieved_ids,relevant_ids,precision@3,recall@3,hit_rate@3,reciprocal_rank
0,Embeddings,How do I get my money back?,"[0, 3, 9]",[0],0.333333,1.0,1,1.0
1,Embeddings,Where can I change my classes?,"[2, 13, 14]",[2],0.333333,1.0,1,1.0
2,Embeddings,I forgot my login details,"[3, 14, 17]",[3],0.333333,1.0,1,1.0
3,Embeddings,When are tests announced?,"[5, 6, 12]",[5],0.333333,1.0,1,1.0
4,Embeddings,Where can I get emotional support?,"[4, 11, 9]",[4],0.333333,1.0,1,1.0
5,Embeddings,Can I live on campus?,"[6, 8, 18]",[6],0.333333,1.0,1,1.0
6,Embeddings,Where do I report a small injury?,"[11, 4, 12]",[11],0.333333,1.0,1,1.0
7,Embeddings,How can I replace my badge?,"[14, 2, 3]",[14],0.333333,1.0,1,1.0
8,Embeddings,Where can I find my missing backpack?,"[17, 14, 13]",[17],0.333333,1.0,1,1.0
9,Embeddings,How do I travel between campus buildings?,"[8, 18, 6]",[8],0.333333,1.0,1,1.0


In [ ]:
all_retriever_evaluations = pd.concat(
    [tfidf_eval, bm25_eval, semantic_eval],
    ignore_index=True
)

summary = all_retriever_evaluations.groupby("retriever")[[
    "precision@3",
    "recall@3",
    "hit_rate@3",
    "reciprocal_rank"
]].mean()

summary


,precision@3,recall@3,hit_rate@3,reciprocal_rank
retriever,,,,
BM25,0.166667,0.5,0.5,0.456031
Embeddings,0.333333,1.0,1.0,1.000000
TF-IDF,0.166667,0.5,0.5,0.456031


## How to Read the Comparison

If embeddings outperform TF-IDF/BM25, it usually means the dataset contains paraphrases or synonyms.

If BM25 outperforms embeddings, it may mean the task depends heavily on exact words, numbers, IDs, or formal terminology.

Professional retrieval design is not about blindly choosing embeddings.

It is about measuring what works for your data.


# Section 17 — Side-by-Side Top-1 Comparison

Aggregate metrics are useful, but we also need to inspect actual mistakes.

This table shows the top result from each retriever for every query.


In [ ]:
def top_1_document_text(results):
    return results.iloc[0]["document"]

comparison_rows = []

for query, relevant_ids in ground_truth.items():
    tfidf_top = retrieve_top_k_tfidf(query, tfidf_vectorizer, tfidf_matrix, documents, k=1)
    bm25_top = retrieve_top_k_bm25(query, documents, bm25, k=1)
    semantic_top = retrieve_top_k_semantic(query, model, documents, document_embeddings, k=1)

    comparison_rows.append({
        "query": query,
        "relevant_ids": relevant_ids,
        "tfidf_top_id": int(tfidf_top.iloc[0]["document_id"]),
        "bm25_top_id": int(bm25_top.iloc[0]["document_id"]),
        "semantic_top_id": int(semantic_top.iloc[0]["document_id"]),
        "tfidf_top_document": top_1_document_text(tfidf_top),
        "bm25_top_document": top_1_document_text(bm25_top),
        "semantic_top_document": top_1_document_text(semantic_top)
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df


,query,relevant_ids,tfidf_top_id,bm25_top_id,semantic_top_id,tfidf_top_document,bm25_top_document,semantic_top_document
0,How do I get my money back?,[0],0,0,0,Students can request a refund within 14 days of payment by submitting the online finance form.,Students can request a refund within 14 days of payment by submitting the online finance form.,Students can request a refund within 14 days of payment by submitting the online finance form.
1,Where can I change my classes?,[2],3,3,2,Technical support can help students reset account passwords and recover access to the student portal.,Technical support can help students reset account passwords and recover access to the student portal.,"The registration office handles enrollment, course changes, section transfers, and class withdrawal requests."
2,I forgot my login details,[3],0,0,3,Students can request a refund within 14 days of payment by submitting the online finance form.,Students can request a refund within 14 days of payment by submitting the online finance form.,Technical support can help students reset account passwords and recover access to the student portal.
3,When are tests announced?,[5],13,18,5,"Printing services are available in the library, computer lab, and student services center.",Parking permits are issued by campus security after vehicle registration and payment confirmation.,The exam timetable is published on the student portal two weeks before the assessment period begins.
4,Where can I get emotional support?,[4],4,4,4,"Counseling services provide confidential academic, emotional, and personal support for students.","Counseling services provide confidential academic, emotional, and personal support for students.","Counseling services provide confidential academic, emotional, and personal support for students."
5,Can I live on campus?,[6],1,1,6,The library is open from 8 AM to 8 PM on weekdays and from 10 AM to 4 PM on Saturdays.,The library is open from 8 AM to 8 PM on weekdays and from 10 AM to 4 PM on Saturdays.,Student housing applications must be submitted before July 15 through the residence services system.
6,Where do I report a small injury?,[11],0,0,11,Students can request a refund within 14 days of payment by submitting the online finance form.,Students can request a refund within 14 days of payment by submitting the online finance form.,"The medical clinic treats minor injuries, provides first aid, and refers urgent cases to nearby hospitals."
7,How can I replace my badge?,[14],3,3,14,Technical support can help students reset account passwords and recover access to the student portal.,Technical support can help students reset account passwords and recover access to the student portal.,Lost student ID cards can be replaced at the student services desk after identity verification.
8,Where can I find my missing backpack?,[17],16,16,17,"The career center helps students prepare resumes, practice interviews, and find internship opportunities.","The career center helps students prepare resumes, practice interviews, and find internship opportunities.",Lost and found items are stored at campus security for thirty days before disposal.
9,How do I travel between campus buildings?,[8],8,8,8,"Campus shuttle buses transport students between the main building, labs, library, and residence halls.","Campus shuttle buses transport students between the main building, labs, library, and residence halls.","Campus shuttle buses transport students between the main building, labs, library, and residence halls."


## What Students Should Analyze

For each query, ask:

1. Which retriever returned the correct document first?
2. Which retriever failed?
3. Did the failed retriever fail because of vocabulary mismatch?
4. Did it fail because of semantic confusion?
5. Did it fail because the query was ambiguous?


# Section 18 — Embedding Failure Cases

Embeddings are powerful, but they are not magic.

They can be dangerous when exact details matter.


## Failure Case 1 — Numbers

```text
Query:    "refund within 7 days"
Document: "refund within 14 days"
```

The texts are semantically close, but the numeric detail is different.

A semantic retriever may rank the document highly even though the exact answer does not match the query.


## Failure Case 2 — Codes and IDs

```text
Query:    "Form A-17 deadline"
Document: "Form A-18 deadline is next Monday."
```

The phrases are very similar, but the code is different.

Embedding models may treat them as close even when the difference is critical.


## Failure Case 3 — Negation

```text
Query:    "students who are not eligible"
Document: "students who are eligible"
```

The sentences share most words, but the meaning is opposite.

Both lexical and semantic systems may struggle if negation is not handled carefully.


## Failure Case 4 — Domain-Specific Abbreviations

```text
Query:    "CRN update"
Document: "course registration number modification"
```

A general embedding model may or may not understand the abbreviation.

In specialized domains, domain-specific models or query expansion may be needed.


In [ ]:
failure_documents = [
    "Students can request a refund within 14 days of payment.",
    "Students can request a refund within 7 days of payment.",
    "Form A-17 must be submitted before Monday.",
    "Form A-18 must be submitted before Monday.",
    "Eligible students may apply for housing.",
    "Students who are not eligible may appeal the decision."
]

failure_embeddings = model.encode(
    failure_documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

failure_query = "refund within 7 days"

retrieve_top_k_semantic(
    query=failure_query,
    model=model,
    documents=failure_documents,
    document_embeddings=failure_embeddings,
    k=6
)


,document_id,document,score
0,1,Students can request a refund within 7 days of payment.,0.654431
1,0,Students can request a refund within 14 days of payment.,0.629364
2,2,Form A-17 must be submitted before Monday.,0.151991
3,3,Form A-18 must be submitted before Monday.,0.118993
4,5,Students who are not eligible may appeal the decision.,0.093379
5,4,Eligible students may apply for housing.,-0.069268


## Ruthless Lesson

Embeddings are strong for paraphrases and synonyms.

They are weaker when the task requires exact matching of:

- numbers,
- dates,
- codes,
- product names,
- legal terms,
- medical details,
- engineering specifications,
- negation.

This is why many production retrieval systems use hybrid retrieval.


# Section 19 — Hybrid Retrieval

Hybrid retrieval combines lexical retrieval and semantic retrieval.

The basic idea:

```text
BM25 catches exact words, numbers, codes, and names.
Embeddings catch synonyms, paraphrases, and meaning.
```

A simple hybrid score can be written as:

$$
\text{hybrid score}=\alpha \times \text{semantic score} + (1-\alpha) \times \text{lexical score}
$$

Where:

- $\alpha$ controls how much we trust semantic retrieval.
- If $\alpha$ is high, semantic retrieval dominates.
- If $\alpha$ is low, lexical retrieval dominates.


## Why Normalize Scores Before Combining?

BM25 scores and embedding cosine scores are on different scales.

Example:

```text
BM25 score: 4.82
Cosine score: 0.61
```

You cannot combine them fairly without normalization.

We will use min-max normalization:

$$
\text{normalized score}=\frac{x-\min(x)}{\max(x)-\min(x)}
$$

This scales scores to the range 0 to 1.


In [ ]:
def min_max_normalize(scores):
    """
    Normalize scores to the range 0 to 1.
    """
    scores = np.array(scores, dtype=float)
    min_score = scores.min()
    max_score = scores.max()

    if max_score == min_score:
        return np.zeros_like(scores)

    return (scores - min_score) / (max_score - min_score)


def retrieve_top_k_hybrid(query, documents, bm25, model, document_embeddings, alpha=0.7, k=3):
    """
    Retrieve top-k documents using a weighted combination of BM25 and embedding similarity.

    alpha:
        Weight given to semantic similarity.
        1 - alpha is the weight given to BM25.
    """
    tokenized_query = simple_tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_scores_normalized = min_max_normalize(bm25_scores)

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    semantic_scores = cosine_similarity(query_embedding, document_embeddings).flatten()
    semantic_scores_normalized = min_max_normalize(semantic_scores)

    hybrid_scores = (
        alpha * semantic_scores_normalized
        + (1 - alpha) * bm25_scores_normalized
    )

    results = pd.DataFrame({
        "document_id": range(len(documents)),
        "document": documents,
        "bm25_score_norm": bm25_scores_normalized,
        "semantic_score_norm": semantic_scores_normalized,
        "hybrid_score": hybrid_scores
    })

    results = results.sort_values(by="hybrid_score", ascending=False)

    return results.head(k).reset_index(drop=True)


In [ ]:
retrieve_top_k_hybrid(
    query="How do I get my money back?",
    documents=documents,
    bm25=bm25,
    model=model,
    document_embeddings=document_embeddings,
    alpha=0.7,
    k=5
)


,document_id,document,bm25_score_norm,semantic_score_norm,hybrid_score
0,0,Students can request a refund within 14 days of payment by submitting the online finance form.,0.0,1.000000,0.700000
1,3,Technical support can help students reset account passwords and recover access to the student portal.,0.0,0.573776,0.401643
2,9,The scholarship office reviews financial aid applications and merit-based funding requests.,0.0,0.534456,0.374119
3,14,Lost student ID cards can be replaced at the student services desk after identity verification.,0.0,0.487056,0.340939
4,17,Lost and found items are stored at campus security for thirty days before disposal.,0.0,0.483845,0.338692


# Section 20 — Evaluate Hybrid Retrieval

We evaluate multiple values of alpha.

This teaches that hybrid retrieval is a design choice, not a fixed rule.


In [ ]:
hybrid_evaluations = []

for alpha in [0.2, 0.5, 0.8]:
    hybrid_eval = evaluate_retriever(
        retriever_name=f"Hybrid alpha={alpha}",
        retrieval_function=lambda query, k, alpha=alpha: retrieve_top_k_hybrid(
            query=query,
            documents=documents,
            bm25=bm25,
            model=model,
            document_embeddings=document_embeddings,
            alpha=alpha,
            k=k
        ),
        ground_truth=ground_truth,
        k=3,
        max_results=len(documents)
    )

    hybrid_evaluations.append(hybrid_eval)

hybrid_eval_df = pd.concat(hybrid_evaluations, ignore_index=True)
hybrid_eval_df


,retriever,query,retrieved_ids,relevant_ids,precision@3,recall@3,hit_rate@3,reciprocal_rank
0,Hybrid alpha=0.2,How do I get my money back?,"[0, 3, 9]",[0],0.333333,1.0,1,1.000000
1,Hybrid alpha=0.2,Where can I change my classes?,"[14, 3, 0]",[2],0.000000,0.0,0,0.250000
2,Hybrid alpha=0.2,I forgot my login details,"[3, 14, 17]",[3],0.333333,1.0,1,1.000000
3,Hybrid alpha=0.2,When are tests announced?,"[18, 13, 17]",[5],0.000000,0.0,0,0.200000
4,Hybrid alpha=0.2,Where can I get emotional support?,"[4, 3, 0]",[4],0.333333,1.0,1,1.000000
5,Hybrid alpha=0.2,Can I live on campus?,"[1, 5, 8]",[6],0.000000,0.0,0,0.100000
6,Hybrid alpha=0.2,Where do I report a small injury?,"[0, 11, 4]",[11],0.333333,1.0,1,0.500000
7,Hybrid alpha=0.2,How can I replace my badge?,"[14, 3, 0]",[14],0.333333,1.0,1,1.000000
8,Hybrid alpha=0.2,Where can I find my missing backpack?,"[16, 14, 3]",[17],0.000000,0.0,0,0.200000
9,Hybrid alpha=0.2,How do I travel between campus buildings?,"[8, 19, 18]",[8],0.333333,1.0,1,1.000000


In [ ]:
final_evaluation_df = pd.concat(
    [tfidf_eval, bm25_eval, semantic_eval, hybrid_eval_df],
    ignore_index=True
)

final_summary = final_evaluation_df.groupby("retriever")[[
    "precision@3",
    "recall@3",
    "hit_rate@3",
    "reciprocal_rank"
]].mean().sort_values(by="reciprocal_rank", ascending=False)

final_summary


,precision@3,recall@3,hit_rate@3,reciprocal_rank
retriever,,,,
Embeddings,0.333333,1.000000,1.000000,1.000000
Hybrid alpha=0.8,0.333333,1.000000,1.000000,0.958333
Hybrid alpha=0.5,0.250000,0.750000,0.750000,0.702183
Hybrid alpha=0.2,0.222222,0.666667,0.666667,0.645833
BM25,0.166667,0.500000,0.500000,0.456031
TF-IDF,0.166667,0.500000,0.500000,0.456031


## How to Choose Alpha

There is no universal best alpha.

Use a validation set.

If queries depend heavily on synonyms and paraphrases, a higher alpha may work better.

If queries depend on exact numbers, codes, names, or IDs, a lower alpha may be safer.

Production systems often tune this using real queries and human relevance labels.


# Section 21 — FAISS Vector Indexing

For 20 documents, comparing a query against every document is fine.

For 1 million documents, brute-force comparison becomes expensive.

A vector index helps retrieve nearest vectors efficiently.

FAISS is a library for similarity search over dense vectors.



In [ ]:
# Run this only if faiss-cpu is installed.
# If it is not installed, run:
# !pip install -q faiss-cpu

import faiss


## Inner Product and Cosine Similarity

FAISS `IndexFlatIP` uses inner product.

If vectors are normalized to length 1, then:

```text
inner product = cosine similarity
```

We already created normalized embeddings using:

```python
normalize_embeddings=True
```


In [ ]:
embedding_dimension = document_embeddings.shape[1]

faiss_index = faiss.IndexFlatIP(embedding_dimension)
faiss_index.add(document_embeddings.astype("float32"))

print("Number of vectors in index:", faiss_index.ntotal)


Number of vectors in index: 20


In [ ]:
def retrieve_top_k_faiss(query, model, documents, faiss_index, k=3):
    """
    Retrieve top-k documents using a FAISS inner-product index.
    """
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_embedding, k)

    results = pd.DataFrame({
        "document_id": indices[0],
        "document": [documents[i] for i in indices[0]],
        "score": scores[0]
    })

    return results


In [ ]:
retrieve_top_k_faiss(
    query="How do I get my money back?",
    model=model,
    documents=documents,
    faiss_index=faiss_index,
    k=5
)


,document_id,document,score
0,0,Students can request a refund within 14 days of payment by submitting the online finance form.,0.452975
1,3,Technical support can help students reset account passwords and recover access to the student portal.,0.173491
2,9,The scholarship office reviews financial aid applications and merit-based funding requests.,0.147708
3,14,Lost student ID cards can be replaced at the student services desk after identity verification.,0.116626
4,17,Lost and found items are stored at campus security for thirty days before disposal.,0.114521


## FAISS Is Not a Full Vector Database

FAISS is a vector index library.

A full vector database usually also provides:

- persistent storage,
- metadata filtering,
- access control,
- APIs,
- collection management,
- distributed scaling,
- backup and monitoring.

So this section teaches vector indexing, not full vector database operations.


# Section 22 — Final Error Analysis

Metrics tell you **how much** the system worked.

Error analysis tells you **why** it failed.

For each retriever, find at least one failed query.


In [ ]:
def find_failed_queries(evaluation_df, k=3):
    """
    Return queries where Hit Rate@K is 0.
    """
    hit_rate_column = f"hit_rate@{k}"
    return evaluation_df[evaluation_df[hit_rate_column] == 0]

find_failed_queries(final_evaluation_df, k=3)


,retriever,query,retrieved_ids,relevant_ids,precision@3,recall@3,hit_rate@3,reciprocal_rank
1,TF-IDF,Where can I change my classes?,"[3, 14, 0]",[2],0.0,0.0,0,0.052632
2,TF-IDF,I forgot my login details,"[0, 1, 18]",[3],0.0,0.0,0,0.055556
3,TF-IDF,When are tests announced?,"[13, 18, 17]",[5],0.0,0.0,0,0.062500
5,TF-IDF,Can I live on campus?,"[1, 5, 3]",[6],0.0,0.0,0,0.058824
6,TF-IDF,Where do I report a small injury?,"[0, 1, 18]",[11],0.0,0.0,0,0.100000
8,TF-IDF,Where can I find my missing backpack?,"[16, 3, 14]",[17],0.0,0.0,0,0.142857
13,BM25,Where can I change my classes?,"[3, 14, 0]",[2],0.0,0.0,0,0.052632
14,BM25,I forgot my login details,"[0, 1, 18]",[3],0.0,0.0,0,0.055556
15,BM25,When are tests announced?,"[18, 13, 17]",[5],0.0,0.0,0,0.062500
17,BM25,Can I live on campus?,"[1, 5, 3]",[6],0.0,0.0,0,0.058824


## Error Analysis Template

For each failed query, answer:

1. What was the query?
2. What document should have been retrieved?
3. What did the system retrieve instead?
4. Was the failure caused by vocabulary mismatch?
5. Was the failure caused by semantic confusion?
6. Was the query ambiguous?
7. Was the correct document missing or poorly written?
8. Would lexical, semantic, or hybrid retrieval be safer?


# Section 23 — Final Lab Assignment

You will now build and evaluate your own semantic retrieval system.


## Task 1 — Build a Corpus

Create at least 20 neutral documents from one domain.

Allowed examples:

- university services,
- hospital FAQ,
- hotel policies,
- software documentation,
- library help desk,
- transportation information,
- customer support knowledge base.

Do not use random unrelated sentences.
The documents must belong to one realistic knowledge base.


## Task 2 — Create Queries

Create at least 12 queries.

At least 6 queries must use different wording from the relevant document.

Example:

```text
Document: "Customers may request a refund within 14 days."
Query:    "How do I get my money back?"
```


## Task 3 — Create Ground Truth

For each query, manually define the relevant document IDs.

Example:

```python
ground_truth = {
    "How do I get my money back?": [0],
    "Where can I reset my account access?": [3]
}
```


## Task 4 — Build Three Retrievers

You must implement:

1. TF-IDF retriever
2. BM25 retriever using `rank-bm25`
3. Embedding retriever using `SentenceTransformer`


## Task 5 — Evaluate the Retrievers

Evaluate all retrievers using:

- Precision@3
- Recall@3
- Hit Rate@3
- Reciprocal Rank
- MRR


## Task 6 — Build Hybrid Retrieval

Combine BM25 and semantic scores.

Try at least three alpha values:

```python
alpha_values = [0.2, 0.5, 0.8]
```

Report which alpha worked best and explain why.


## Task 7 — Error Analysis

Choose at least 3 failed queries.

For each one, explain:

1. The query.
2. The correct document.
3. The wrong document retrieved.
4. Why the retriever failed.
5. Whether TF-IDF, BM25, embeddings, or hybrid retrieval handled it best.


# Final Ruthless Takeaways

1. TF-IDF and BM25 are lexical retrievers.
2. Lexical retrievers are strong when exact words matter.
3. Embeddings are semantic retrievers.
4. Semantic retrievers are strong when meaning matters more than word overlap.
5. Embeddings can fail on numbers, codes, negation, and exact facts.
6. Hybrid retrieval often performs better than either lexical or semantic retrieval alone.
7. Retrieval must be evaluated before building RAG.
8. Bad retrieval creates bad generated answers.


# References and Further Reading

- Sentence Transformers semantic search documentation: https://sbert.net/examples/sentence_transformer/applications/semantic-search/README.html
- Sentence Transformers documentation: https://sbert.net/
- rank-bm25 package: https://pypi.org/project/rank-bm25/
- rank-bm25 GitHub repository: https://github.com/dorianbrown/rank_bm25
- FAISS documentation: https://faiss.ai/
